# rxn_core Minimal Pipeline Tutorial

This notebook runs a concrete direct-XYZ example: `pr1.tempo_ts3` from the appendix benchmark. The benchmark step layout is only a wrapper; the principal inputs are molecule XYZ files plus charge and multiplicity.

Values marked `# default to ...` are internal defaults. They are shown because they are algorithm hypotheses, but usually do not need to be changed.


## Import


In [1]:
from pathlib import Path
import rxn_core.pipeline as rxnp


## Real Example Inputs And Output Location

This example uses one complete reactant-complex XYZ, one complete product-complex XYZ, one IG, and one optional GT TS. All generated files are stored under `docs/example_runs/pr1.tempo_ts3/`.


In [2]:
benchmark_root = Path("~/Downloads/appendix_final 2/benchmark").expanduser()
step_dir = benchmark_root / "pr1.tempo_ts3"

name = "pr1.tempo_ts3"
reactant_xyz = step_dir / "reactants/reactant_01_reactant_01_5.xyz"
product_xyz = step_dir / "products/product_01_product_01_6.xyz"

docs_dir = Path.cwd() if Path.cwd().name == "docs" else Path("docs")
doc_run_root = docs_dir / "example_runs" / name
workdir = doc_run_root / "work"
rxnp.STAGE_ROOT = doc_run_root / "stages"
rxnp.OUT_ROOT = doc_run_root / "views"
rxnp.ALIGNMENT_OUT_ROOT = doc_run_root / "alignments"
rxnp.EVAL_JSON = doc_run_root / "eval.json"

target_specs = [
    {
        "kind": "ig",
        "label": "iter1",
        "xyz": step_dir / "initial_guess/pr1.tempo_ts3_benchmark_plain_iter1_87ea3b8f.xyz",
    },
    {
        "kind": "gt",
        "label": "GT",
        "xyz": step_dir / "groundtruth/ts_groundtruth_01_reference_ts_01_TS3.xyz",
    },
]

charge = 0        # default to 0; this example XYZ comment also says charge=0
multiplicity = 1  # default to 1; this example XYZ comment also says multiplicity=1
workers = 8       # choose for your machine/job
xtb_mode = "auto" # default to "auto"; use "cache-only" to forbid xtb runs
save_alignment_files = True  # default to False
mechanism_ids = None         # default to None; set e.g. [1] to verify selected mechanisms


## Stage 1 Hypotheses: R-P Alignment And Mechanism Discovery

These control sweep-cut alignment, symmetry handling, and broken/forming bond classification.


In [3]:
rp_config = {
    "cut_floor": 0.2,                    # default to 0.2
    "graph_floor": 0.2,                  # default to 0.2
    "iso_tol": 1.0,                      # default to 1.0
    "dwbo_threshold": 0.5,               # default to 0.5
    "symmetry_wbo_tol": 0.2,             # default to 0.2
    "n_seeds": 3,                        # default to 3
    "max_branches": 100,                 # default to 100
    "chunksize": 1,                      # default to 1
    "symmetry_repair": True,             # default to True
    "symmetry_repair_min_changes": 5,    # default to 5
    "symmetry_repair_max_evals": 20000,  # default to 20000
}


## Stage 2 Hypotheses: TS / IG / GT Verification

These control endpoint-to-TS core matching and final imaginary-mode scoring.


In [4]:
ts_config = {
    "iso_tol": 1.0,             # default to 1.0
    "edge_floor": 0.2,          # default to 0.2
    "max_candidates": 20000,    # default to 20000
    "score": {
        "W_RXN": 1.0,           # default to 1.0
        "W_CORE": 0.2,          # default to 0.2
        "IMAG_PEN": 0.3,        # default to 0.3
    },
}


## Run Stage 1

Outputs to check after this cell:

- `docs/example_runs/pr1.tempo_ts3/stages/pr1.tempo_ts3/rp_stage.json`
- `docs/example_runs/pr1.tempo_ts3/views/pr1.tempo_ts3/view.html`
- `docs/example_runs/pr1.tempo_ts3/alignments/pr1.tempo_ts3/`


In [5]:
rp_stage = rxnp.process_xyz_stage(
    name,
    reactant_xyz,
    product_xyz,
    workdir=workdir,
    stage="rp",
    charge=charge,
    multiplicity=multiplicity,
    xtb_mode=xtb_mode,
    inner_workers=workers,
    save_alignment_files=save_alignment_files,
    rp_config=rp_config,
)

{
    "n_mechanisms": rp_stage["slim"]["n_mechs"],
    "mechanisms": [
        {
            "id": mech["id"],
            "broken_R": mech["broken_R"],
            "formed_R": mech["formed_R"],
        }
        for mech in rp_stage["slim"]["mechanisms"]
    ],
    "rp_stage_json": str(rxnp.pipeline_stage_paths(name).rp_json),
    "view_html": rp_stage["view"]["view_html"],
    "alignment_manifest": rp_stage["alignment_files"]["manifest"],
}


{'n_mechanisms': 1,
 'mechanisms': [{'id': 1, 'broken_R': [(12, 49)], 'formed_R': [(12, 14)]}],
 'rp_stage_json': '/Users/yunhengz/codex_AAM/rxn_core/docs/example_runs/pr1.tempo_ts3/stages/pr1.tempo_ts3/rp_stage.json',
 'view_html': '/Users/yunhengz/codex_AAM/rxn_core/docs/example_runs/pr1.tempo_ts3/views/pr1.tempo_ts3/view.html',
 'alignment_manifest': '/Users/yunhengz/codex_AAM/rxn_core/docs/example_runs/pr1.tempo_ts3/alignments/pr1.tempo_ts3/manifest.json'}

## Run Stage 2

Stage 2 resumes from Stage 1 and scores the IG/GT targets. Outputs to check after this cell:

- `docs/example_runs/pr1.tempo_ts3/stages/pr1.tempo_ts3/ts_stage.json`
- `docs/example_runs/pr1.tempo_ts3/views/pr1.tempo_ts3/view.html`
- `docs/example_runs/pr1.tempo_ts3/alignments/pr1.tempo_ts3/ts_alignments/`


In [6]:
ts_stage = rxnp.process_xyz_stage(
    name,
    reactant_xyz,
    product_xyz,
    workdir=workdir,
    stage="ts",
    target_specs=target_specs,
    mechanism_ids=mechanism_ids,
    charge=charge,
    multiplicity=multiplicity,
    xtb_mode=xtb_mode,
    inner_workers=workers,
    save_alignment_files=save_alignment_files,
    ts_config=ts_config,
)

{
    "n_mechanisms": ts_stage["slim"]["n_mechs"],
    "mechanisms": [
        {
            "id": mech["id"],
            "broken_R": mech["broken_R"],
            "formed_R": mech["formed_R"],
            "GT_S": None if mech.get("gt") is None else mech["gt"]["S"],
            "IG_scores": [
                {"label": ig["label"], "S": ig.get("S")}
                for ig in mech.get("igs", [])
            ],
        }
        for mech in ts_stage["slim"]["mechanisms"]
    ],
    "ts_stage_json": str(rxnp.pipeline_stage_paths(name).ts_json),
    "view_html": ts_stage["view"]["view_html"],
    "ts_alignment_manifest": ts_stage["ts_alignment_files"]["manifest"],
}


{'n_mechanisms': 1,
 'mechanisms': [{'id': 1,
   'broken_R': [[12, 49]],
   'formed_R': [[12, 14]],
   'GT_S': 1.608575941081719,
   'IG_scores': [{'label': 'iter1', 'S': 1.76895599877841}]}],
 'ts_stage_json': '/Users/yunhengz/codex_AAM/rxn_core/docs/example_runs/pr1.tempo_ts3/stages/pr1.tempo_ts3/ts_stage.json',
 'view_html': '/Users/yunhengz/codex_AAM/rxn_core/docs/example_runs/pr1.tempo_ts3/views/pr1.tempo_ts3/view.html',
 'ts_alignment_manifest': '/Users/yunhengz/codex_AAM/rxn_core/docs/example_runs/pr1.tempo_ts3/alignments/pr1.tempo_ts3/ts_alignments/manifest.json'}

## Full Pipeline Alternative

Use this instead of the two separate stage calls when you want mechanism discovery, TS verification, and the viewer in one call:

```python
full = rxnp.process_xyz_stage(
    name,
    reactant_xyz,
    product_xyz,
    workdir=workdir,
    stage="full",
    target_specs=target_specs,
    mechanism_ids=mechanism_ids,
    charge=charge,
    multiplicity=multiplicity,
    xtb_mode=xtb_mode,
    inner_workers=workers,
    save_alignment_files=save_alignment_files,
    rp_config=rp_config,
    ts_config=ts_config,
)
```


## Command Line Equivalents

```bash
rxn-core --stage rp --name pr1.tempo_ts3   --reactant-xyz "~/Downloads/appendix_final 2/benchmark/pr1.tempo_ts3/reactants/reactant_01_reactant_01_5.xyz"   --product-xyz "~/Downloads/appendix_final 2/benchmark/pr1.tempo_ts3/products/product_01_product_01_6.xyz"   --workdir docs/example_runs/pr1.tempo_ts3/work   --charge 0 --multiplicity 1 --workers 8

rxn-core --stage ts --name pr1.tempo_ts3   --reactant-xyz "~/Downloads/appendix_final 2/benchmark/pr1.tempo_ts3/reactants/reactant_01_reactant_01_5.xyz"   --product-xyz "~/Downloads/appendix_final 2/benchmark/pr1.tempo_ts3/products/product_01_product_01_6.xyz"   --target-xyz "~/Downloads/appendix_final 2/benchmark/pr1.tempo_ts3/initial_guess/pr1.tempo_ts3_benchmark_plain_iter1_87ea3b8f.xyz"   --target-label iter1 --target-kind ig   --target-xyz "~/Downloads/appendix_final 2/benchmark/pr1.tempo_ts3/groundtruth/ts_groundtruth_01_reference_ts_01_TS3.xyz"   --target-label GT --target-kind gt   --workdir docs/example_runs/pr1.tempo_ts3/work   --charge 0 --multiplicity 1 --workers 8
```

CLI flags expose the common hypothesis knobs: `--iso-tol`, `--dwbo-threshold`, `--symmetry-wbo-tol`, `--w-rxn`, `--w-core`, and `--imag-pen`. The Python config dictionaries above expose the full set.


## Prepared Step Folder And Benchmark Step Adapter

The executed direct-XYZ run also stores a prepared cache-only step folder:

```text
docs/example_runs/pr1.tempo_ts3/prepared_steps/pr1.tempo_ts3/
  R/
  P/
  sp_iter1/
  hess_iter1/
  sp_groundtruth/
  hess_groundtruth/
```

That folder is the wrapper layout consumed by `load_step_inputs(...)` and `load_ts_targets(...)`:

```python
rxnp.WORK = doc_run_root / "prepared_steps"
inputs = rxnp.load_step_inputs(name, xtb_mode="cache-only")
rp = rxnp.run_rp_stage(inputs, config=rp_config, inner_workers=workers)
targets = rxnp.load_ts_targets(inputs, include_gt=True)
ts = rxnp.run_ts_stage(
    inputs,
    rp,
    targets,
    config=ts_config,
    mechanism_ids=mechanism_ids,
    inner_workers=workers,
)
view = rxnp.write_view_stage(inputs, rp, ts, include_gt=True)
```
